# FastVQ Quickstart

This notebook shows basic quantize/dequantize usage, arbitrary trailing dimensions, compression stats, and synthetic benchmarks.


In [ ]:
import numpy as np
from fastvq import TurboQuant, TurboQuantConfig, compute_distortion

rng = np.random.default_rng(42)
x = rng.standard_normal((256, 192)).astype(np.float32)
config = TurboQuantConfig(bit_width=3, block_size=128, rotation='hadamard')
quantizer = TurboQuant(config)
quantized = quantizer.quantize(x)
reconstructed = quantizer.dequantize(quantized)

print(reconstructed.shape)
print(quantizer.compression_stats(quantized))
print(compute_distortion(x, reconstructed))


In [ ]:
payload = quantizer.compress(x)
roundtrip = TurboQuant().decompress(payload)
print(len(payload), roundtrip.shape, np.mean((x - roundtrip) ** 2))


In [ ]:
from fastvq import run_benchmark_suite

results = run_benchmark_suite(
    shapes=[(1024, 128), (1024, 192)],
    bit_widths=(2, 3, 4),
    block_sizes=(128,),
    trials=3,
)
for row in results:
    print(row['shape'], row['bit_width'], round(row['compression_ratio'], 2), round(row['cosine_similarity'], 4))
